In [1]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [2]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [3]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [96]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [7]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [8]:
chain = prompt | llm | output_parser

## NER Model

In [9]:
import spacy
from span_marker import SpanMarkerModel

In [11]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

config.json:   0%|          | 0.00/5.45k [00:00<?, ?B/s]

C:\Users\axel0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\axel0\.cache\huggingface\hub\models--tomaarsen--span-marker-roberta-large-ontonotes5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(messag

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

C:\Users\axel0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\axel0\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Exception ignored in: <function

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Google Maps

In [16]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [17]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [18]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

## Pipeline Entry Point

In [266]:
# Temporary. Use given article data set
full_df = pd.read_csv("./sample_data/Articles_Nov_2020_March_2023.csv")

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 10 articles
raw_df = full_df.sample(10)
# raw_df = full_df
len(raw_df)

10

In [267]:
# Uncomment later 
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [268]:
# raw_df = pd.read_csv(sample_data_dir)

In [269]:
raw_df.head(3)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,Article,LIVESTREAM: Boston Public Radio live from the ...,LIVESTREAM: Boston Public Radio live from the ...,Boston Public Radio Staff,NaN,Local News,NaN,/local-news/2022/04/05/livestream-boston-publi...,Tue Apr 05 12:13:57 EDT 2022,TRUE,"<iframe src=""https://player.wgbh.org/media.php..."
11325,00000185-0634-d270-adb5-57b7d9440001,Article,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass is sworn in as Los Angeles mayor as...,Ayana Archie,NaN,National News,NaN,/national-news/2022/12/12/karen-bass-is-sworn-...,Mon Dec 12 06:36:00 EST 2022,TRUE,Karen Bass was sworn in as the mayor of Los An...
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,Article,More than 2 million baby swings and rockers ar...,More than 2 million baby swings and rockers ar...,Wynne Davis,NaN,National News,NaN,/national-news/2022/08/16/more-than-2-million-...,Tue Aug 16 10:24:00 EDT 2022,TRUE,4moms is recalling more than 2 million of its ...


The ML Model honestly just needs the `id`, `header`, and `body`.

In [271]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [272]:
# For Testing Purposes Only
# df = df[:20]

In [273]:
df["llama_prediction"] = None # Add the llama_prediction

Remove Duplicates (if any)

In [274]:
duplicates = df.duplicated(subset=['hl1'])

In [275]:
print(duplicates.value_counts())

False    10
Name: count, dtype: int64


In [276]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header -> Regex Cleaner

In [277]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

100%|██████████| 10/10 [00:00<?, ?it/s]


In [278]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 10/10 [00:00<?, ?it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary

In [279]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    known_locs_path = "./geodata/known_locs.json"
    # Load known locations of the form (location: [lat, long])
    with open(known_locs_path, 'r') as file:
        known_locs_dict = json.load(file) 
    
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_locs_dict.keys():
        # if (key in lowercase_header):
        if (location.lower() in lowercase_header):
            return [location, known_locs_dict[location]]   
    return None

In [406]:
df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

100%|██████████| 10/10 [00:00<00:00, 175.90it/s]


In [407]:
df["Explicit_Pass"].value_counts().head(10)

Explicit_Pass
[Boston Public Radio, {'lat': 42.3600825, 'lon': -71.0588801}]    1
Name: count, dtype: int64

In [409]:
# Extract the coordinates from the explicit pass
def extract_coordinates(entities):
    if entities is None:
        return None
    
    coordinates = entities[1]
    latitude = coordinates['lat']
    longitude = coordinates['lon']
    formatted_coordinates = [str(longitude), str(latitude)]
    return formatted_coordinates

df['Explicit_Pass_Coordinates'] = df["Explicit_Pass"].progress_apply(extract_coordinates)

100%|██████████| 10/10 [00:00<?, ?it/s]


In [411]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass_1,NER_Pass_1,NER_Pass_1_Sorted,NER_Pass_1_Coordinates,NER_prediction,NER_Sorted,NER_Pred_Sorted,NER_Pred_Coordinates,Tracts,Explicit_Pass,Explicit_Pass_Coordinates
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,LIVESTREAM Boston Public Radio live from the G...,,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...",None,None,None,None,None,None,None,[No TRACT found],"[Boston Public Radio, {'lat': 42.3600825, 'lon...","[-71.0588801, 42.3600825]"
11325,00000185-0634-d270-adb5-57b7d9440001,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass was sworn in as the mayor of Los An...,Based on the information provided in the art...,None,"[((Karen, Bass), PERSON), ((Los, Angeles), GPE...","[((Los, Angeles, City, Council), ORG), ((NPR),...",None,"[((1), CARDINAL), ((Los, Angeles), GPE), ((Los...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[-70.9078346, 42.3020647]",[500101],None,None
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,More than million baby swings and rockers are ...,4moms is recalling more than million of its Ma...,Response:\n\n1. Y - The article is talking a...,None,"[((4moms), ORG), ((more, than, million), CARDI...","[((4moms), ORG), ((the, Consumer, Product, Saf...",None,"[((1), CARDINAL), ((Boston), GPE), ((Massachus...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[-71.3824374, 42.4072107]",[365100],None,None


### NER Code First Pass

In [284]:
# Run NER on the body of the article and return all entities
def predict_NER_def(text):
    try:
        if (text == None or text == ""):
            return None
        
        # Extract entities from the text
        entities = []
        for entity in nlp(text).ents:
            entities.append((entity.text, entity.label_))

        return entities
    except Exception as error:
        print(error) # Double prints error?
        return None

In [285]:
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return predict_NER_def(article['body'])
    except Exception as error:
        print(error)
        return None

In [286]:
df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)

  0%|          | 0/10 [00:00<?, ?it/s]

Passed on LIVESTREAM Boston Public Radio live from the GBH studio at the Boston Public Library


100%|██████████| 10/10 [17:35<00:00, 105.54s/it]


Filter based on superb specific places such as 'FAC'.

In [386]:
# Filter out the entities that are not locations and sort them by priorities
def filter_locations(entities):

    if (entities == None):
        return None
    
    locations = []
    for entity in entities:
        # Check if entity is of the format (location, label)
        if (len(entity) >= 2): 
            location, label = entity
            
            # Attempt to make location prettier
            # print(type(location))
            # if type(location) == tuple:
            #     location = (" ".join(location))
            #     print(location)
            #     print(type(location))
            
            # If the entity is a location, add it to the list
            if (("GPE" in label 
                    and "Boston" not in location 
                    and "Massachusetts" not in location) # Geopolitical Entity but not general like Boston / MA
                 or ("ORG" in label) # Organization
                 or ("FAC" in label) # Facility
                 or ("LOC" in label) # Location
                ):
                locations.append((location, label.strip()))
    
    # Sort the locations by priority
    priority_order = {'FAC': 1, 'ORG': 2, 'LOC': 3, 'GPE': 4}
    sorted_locations = sorted(locations, key=lambda entity: priority_order[entity[1]])
    
    return sorted_locations

In [387]:
df['NER_Pass_Sorted'] = df['NER_Pass'].progress_apply(filter_locations)

100%|██████████| 10/10 [00:00<?, ?it/s]


In [348]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass_1,NER_Pass_1,NER_Pass_1_Sorted,NER_Pass_1_Coordinates
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,LIVESTREAM Boston Public Radio live from the G...,,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...",None,None,None
11325,00000185-0634-d270-adb5-57b7d9440001,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass was sworn in as the mayor of Los An...,None,None,"[((Karen, Bass), PERSON), ((Los, Angeles), GPE...","[((Los, Angeles, City, Council), ORG), ((NPR),...",None
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,More than million baby swings and rockers are ...,4moms is recalling more than million of its Ma...,None,None,"[((4moms), ORG), ((more, than, million), CARDI...","[((4moms), ORG), ((the, Consumer, Product, Saf...",None


Then we get the coordinates through the first pass

In [396]:
# Get the coordinates of the location
def getCoordinates(entities, valid_labels):
    # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction

    if (entities == None or len(entities) == 0):
        return None  
    
    # Get the first entity. Since it was sorted by priority, the first location should be the most specific
    entity = entities[0]
    location, label = entity

    # Check if the location is a valid label, if not, return None
    if (label in valid_labels):
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
        
        # locations are limited to Massachusetts for now
        response = gmaps.geocode(location, components={"administrative_area_level": "MA", "country": "US"})
    else:
        return None 

    if (len(response) == 0):
        return None  
    
    latitude = response[0]['geometry']['location']['lat']
    longitude = response[0]['geometry']['location']['lng']
    
    return [longitude, latitude]

In [378]:
df['NER_Pass_Coordinates'] = df['NER_Pass_Sorted'].progress_apply(getCoordinates, valid_labels=['FAC'])

100%|██████████| 10/10 [00:00<00:00, 104.16it/s]

Location: Los Angeles City Council
Location: 4moms
Location: Residential Assistance for Families in Transition
Location: Capitol
Location: Miss America
Location: GBH Create
Location: the University of North Dakota
Location: McLean Hospital
Location: Disney


In [351]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass_1,NER_Pass_1,NER_Pass_1_Sorted,NER_Pass_1_Coordinates
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,LIVESTREAM Boston Public Radio live from the G...,,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...",None,None,None
11325,00000185-0634-d270-adb5-57b7d9440001,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass was sworn in as the mayor of Los An...,None,None,"[((Karen, Bass), PERSON), ((Los, Angeles), GPE...","[((Los, Angeles, City, Council), ORG), ((NPR),...",None
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,More than million baby swings and rockers are ...,4moms is recalling more than million of its Ma...,None,None,"[((4moms), ORG), ((more, than, million), CARDI...","[((4moms), ORG), ((the, Consumer, Product, Saf...",None


### Llama Prediction

In [352]:
# Run the LLM model on the articles that haven't been tagged with a location yet
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Pass_Coordinates'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            return chain.invoke({"headline": article['hl1'], "body": article['body']})
    except Exception as error:
        print(error)
        return None

In [353]:
df['llama_prediction'] = df.progress_apply(predict_llama, axis=1)

  0%|          | 0/10 [00:00<?, ?it/s]

Has location from title or NER: LIVESTREAM Boston Public Radio live from the GBH studio at the Boston Public Library
  Based on the information provided in the article, here is my response:
1. Y - The article is talking about a specific location within Los Angeles, specifically the city of Los Angeles.
2. Specific location within the city: According to the article, Karen Bass was sworn in as the mayor of Los Angeles on Sunday, which means the article is referring to the city of Los Angeles.
3. Involved specific locations or organizations explicitly found within the article that influenced my decision:
* The article mentions the location of City Hall where protests have taken place following the release of recordings featuring racist remarks by city council members.
* The article also mentions the county of Los Angeles, which has seen a rise in homelessness since the pandemic.
Based on these specific locations and organizations mentioned in the article, I believe the article is talking 


llama_print_timings:        load time =   42655.49 ms
llama_print_timings:      sample time =      65.92 ms /   198 runs   (    0.33 ms per token,  3003.73 tokens per second)
llama_print_timings: prompt eval time =   42627.35 ms /   639 tokens (   66.71 ms per token,    14.99 tokens per second)
llama_print_timings:        eval time =   34281.28 ms /   197 runs   (  174.02 ms per token,     5.75 tokens per second)
llama_print_timings:       total time =   77548.12 ms /   836 tokens
 30%|███       | 3/10 [01:17<03:01, 25.94s/it]Llama.generate: prefix-match hit


  Response:

1. Y - The article is talking about a specific location, Boston, Massachusetts.
2. The specific location within Boston where the incident occurred is the city's BuyBuy Baby and Target stores as well as online through Amazon and the 4moms website.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* BuyBuy Baby
* Target
* Amazon
* 4moms (the company that manufactured the recalled products)

Based on the information provided in the article, it appears that the recall affects a specific geographic area of Boston where the products were sold at these locations. The article also mentions that a smaller number of products were sold in Canada, but it does not provide any specific locations in Canada. Therefore, I have focused on the Boston-specific locations mentioned in the article.


llama_print_timings:        load time =   42655.49 ms
llama_print_timings:      sample time =      61.46 ms /   189 runs   (    0.33 ms per token,  3075.07 tokens per second)
llama_print_timings: prompt eval time =   33069.49 ms /   453 tokens (   73.00 ms per token,    13.70 tokens per second)
llama_print_timings:        eval time =   33606.80 ms /   188 runs   (  178.76 ms per token,     5.59 tokens per second)
llama_print_timings:       total time =   67131.66 ms /   641 tokens
 40%|████      | 4/10 [02:24<03:55, 39.23s/it]Llama.generate: prefix-match hit


...


llama_print_timings:        load time =   42655.49 ms
llama_print_timings:      sample time =      75.88 ms /   247 runs   (    0.31 ms per token,  3255.14 tokens per second)
llama_print_timings: prompt eval time =   77782.35 ms /   992 tokens (   78.41 ms per token,    12.75 tokens per second)
llama_print_timings:        eval time =   49040.41 ms /   246 runs   (  199.35 ms per token,     5.02 tokens per second)
llama_print_timings:       total time =  127426.97 ms /  1238 tokens
 90%|█████████ | 9/10 [08:28<01:09, 69.35s/it]Llama.generate: prefix-match hit


  1. Y - Yes, the article is talking about a region of Boston.
2. Based on the information provided in the article, I believe the specific location within Boston being referred to is Belmont.
3. The following locations or organizations are explicitly mentioned in the article as influencing the discussion on addressing mental health concerns in the restaurant industry:
* McLean Hospital in Belmont
* RISE Hospitality
* The McLean Institute for Technology in Psychiatry
* MIDA (mentioned as the restaurant where chef and owner Douglass Williams works)


llama_print_timings:        load time =   42655.49 ms
llama_print_timings:      sample time =      39.41 ms /   123 runs   (    0.32 ms per token,  3121.19 tokens per second)
llama_print_timings: prompt eval time =   14805.05 ms /   195 tokens (   75.92 ms per token,    13.17 tokens per second)
llama_print_timings:        eval time =   21305.32 ms /   122 runs   (  174.63 ms per token,     5.73 tokens per second)
llama_print_timings:       total time =   36375.58 ms /   317 tokens
100%|██████████| 10/10 [09:04<00:00, 61.03s/it]Llama.generate: prefix-match hit


  1. Y - The article is talking about a specific location within Boston, specifically Florida.
2. The specific location within the city of Boston is Florida, where the Don Say Gay bill was passed by the Florida Senate and is now awaiting signature from Governor Ron DeSantis.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* Florida Senate
* Florida Representative Joe Harding (R)
* State Senator Dennis Baxley (R)
* Disney
* Animator and director Dana Terrace, who created the Disney animated series The Owl House.
* Geoff Morrell, new Chief Corporate Affairs Officer at Disney.


llama_print_timings:        load time =   42655.49 ms
llama_print_timings:      sample time =      48.40 ms /   147 runs   (    0.33 ms per token,  3037.38 tokens per second)
llama_print_timings: prompt eval time =   44381.64 ms /   558 tokens (   79.54 ms per token,    12.57 tokens per second)
llama_print_timings:        eval time =   25968.31 ms /   146 runs   (  177.87 ms per token,     5.62 tokens per second)
llama_print_timings:       total time =   70699.02 ms /   704 tokens
100%|██████████| 10/10 [10:15<00:00, 61.51s/it]


In [405]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass_1,NER_Pass_1,NER_Pass_1_Sorted,NER_Pass_1_Coordinates,NER_prediction,NER_Sorted,NER_Pred_Sorted,NER_Pred_Coordinates,Tracts
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,LIVESTREAM Boston Public Radio live from the G...,,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...",None,None,None,None,None,None,None,[No TRACT found]
11325,00000185-0634-d270-adb5-57b7d9440001,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass was sworn in as the mayor of Los An...,Based on the information provided in the art...,None,"[((Karen, Bass), PERSON), ((Los, Angeles), GPE...","[((Los, Angeles, City, Council), ORG), ((NPR),...",None,"[((1), CARDINAL), ((Los, Angeles), GPE), ((Los...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[-70.9078346, 42.3020647]",[500101]
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,More than million baby swings and rockers are ...,4moms is recalling more than million of its Ma...,Response:\n\n1. Y - The article is talking a...,None,"[((4moms), ORG), ((more, than, million), CARDI...","[((4moms), ORG), ((the, Consumer, Product, Saf...",None,"[((1), CARDINAL), ((Boston), GPE), ((Massachus...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[-71.3824374, 42.4072107]",[365100]


We then apply NER on the llama outputs

In [355]:
df['NER_Prediction'] = df['llama_prediction'].progress_apply(predict_NER_def)

100%|██████████| 10/10 [03:55<00:00, 23.53s/it]


In [356]:
df

,_id,hl1,body,llama_prediction,Explicit_Pass_1,NER_Pass_1,NER_Pass_1_Sorted,NER_Pass_1_Coordinates,NER_prediction
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,LIVESTREAM Boston Public Radio live from the G...,,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...",None,None,None,None
11325,00000185-0634-d270-adb5-57b7d9440001,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass was sworn in as the mayor of Los An...,Based on the information provided in the art...,None,"[((Karen, Bass), PERSON), ((Los, Angeles), GPE...","[((Los, Angeles, City, Council), ORG), ((NPR),...",None,"[((1), CARDINAL), ((Los, Angeles), GPE), ((Los..."
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,More than million baby swings and rockers are ...,4moms is recalling more than million of its Ma...,Response:\n\n1. Y - The article is talking a...,None,"[((4moms), ORG), ((more, than, million), CARDI...","[((4moms), ORG), ((the, Consumer, Product, Saf...",None,"[((1), CARDINAL), ((Boston), GPE), ((Massachus..."
4320,0000017a-0ca7-d770-a57f-edbf9b070001,We re Just Asking For Some Compassion Tenants ...,When the pandemic first hit Dezyre Lewis found...,None,None,"[((first), ORDINAL), ((Dezyre, Lewis), PERSON)...","[((Residential, Assistance, for, Families, in,...",None,None
1376,00000176-d8be-de44-a77e-febe4a4b0001,Pressley Moulton Call For Trump Removal After ...,As pro Trump extremists began to recede from t...,None,None,"[((Trump), PERSON), ((U.S.), GPE), ((Capitol),...","[((Capitol), FAC), ((Capitol), FAC), ((the, Ho...","[-71.0638101, 42.3587772]",None
8076,0000017f-9d68-d191-a1ff-fd6869480001,We can always strive to be more inclusive Miss...,The Miss America contest has been around for m...,"Based on the article, I would guess that the...",None,"[((Miss, America), WORK_OF_ART), ((more, than,...","[((Miss, America), ORG), ((Miss, America), ORG...",None,"[((Cambridge), GPE), ((Massachusetts), GPE), (..."
5228,0000017b-410a-d5be-a57b-fd8a3b1d0001,Friday August 20,Discover America creative spirit through the l...,Based on the information provided in the art...,None,"[((America), GPE), ((American), NORP), ((Rhian...","[((GBH, Create), ORG), ((Backcredit, Craft, in...",None,"[((Boston), GPE), ((Massachusetts), GPE), ((1)..."
12382,00000186-9287-d649-ab9f-bf87b3830001,Scientists are flying into snowstorms to explo...,High up in some ice filled clouds sitting insi...,"Based on the article, I would guess that the...",None,"[((Christian, Nairy), PERSON), ((Nairy), PERSO...","[((the, University, of, North, Dakota), ORG), ...",None,"[((the, United, States), GPE), ((Virginia), GP..."
4666,0000017a-8140-d2d2-a1fe-d3e93c780001,Addressing Mental Health Concerns In The Resta...,While the pandemic has impacted the mental hea...,"1. Y - Yes, the article is talking about a r...",None,"[((McLean, Hospital), ORG), ((Belmont), GPE), ...","[((McLean, Hospital), ORG), ((RISE, Hospitalit...",None,"[((1), CARDINAL), ((Boston), GPE), ((2), CARDI..."
7916,0000017f-6ae7-d19f-a3ff-eff72efd0001,Disney employees furious the company won denou...,Disney employees are showing their outrage ove...,1. Y - The article is talking about a specif...,None,"[((Disney), ORG), ((Florida), GPE), ((Don, Say...","[((Disney), ORG), ((The, Florida, Senate), ORG...",None,"[((1), CARDINAL), ((Boston), GPE), ((Florida),..."


Sort the NER Predictions

In [393]:
df['NER_Pred_Sorted'] = df['NER_Prediction'].progress_apply(filter_locations)

100%|██████████| 10/10 [00:00<?, ?it/s]


Get the coordinates of the most specific location, which in this case we'll allow both FACilities and ORGanizations as it's the final pass.

In [398]:
df['NER_Pred_Coordinates'] = df['NER_Pred_Sorted'].progress_apply(getCoordinates, valid_labels=['FAC', 'ORG'])

100%|██████████| 10/10 [00:00<00:00, 24.51it/s]


In [399]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass_1,NER_Pass_1,NER_Pass_1_Sorted,NER_Pass_1_Coordinates,NER_prediction,NER_Sorted,NER_Pred_Sorted,NER_Pred_Coordinates
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,LIVESTREAM Boston Public Radio live from the G...,,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...",None,None,None,None,None,None,None
11325,00000185-0634-d270-adb5-57b7d9440001,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass was sworn in as the mayor of Los An...,Based on the information provided in the art...,None,"[((Karen, Bass), PERSON), ((Los, Angeles), GPE...","[((Los, Angeles, City, Council), ORG), ((NPR),...",None,"[((1), CARDINAL), ((Los, Angeles), GPE), ((Los...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[-70.9078346, 42.3020647]"
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,More than million baby swings and rockers are ...,4moms is recalling more than million of its Ma...,Response:\n\n1. Y - The article is talking a...,None,"[((4moms), ORG), ((more, than, million), CARDI...","[((4moms), ORG), ((the, Consumer, Product, Saf...",None,"[((1), CARDINAL), ((Boston), GPE), ((Massachus...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[-71.3824374, 42.4072107]"


## Geocode locations

In [400]:
# Get the census tract of the location
def query_census_api(longitude, latitude):
    county = "NO COUNTY"
    url = f"https://geocoding.geo.census.gov/geocoder/geographies/coordinates?x={longitude}&y={latitude}&benchmark=Public_AR_Current&vintage=Census2020_Current&format=json"
    response = requests.get(url)
    if (response.status_code == 200):
        results = response.json()
        # print(results['result']['geographies'])
        census_tracts = results['result']['geographies'].get('Census Tracts', [])
        # county = results['result']['geographies']['County Subdivisions'][0].get('COUNTY')
        # county = results['result']['geographies'].get('COUNTY', [])

        if (census_tracts):
            return census_tracts[0].get('TRACT', 'No TRACT found'), county # Returning the TRACT of the first census tract found
    return "No TRACT found", county  # Return this if API call failed or no tracts found

In [401]:
# Get the census tract of the location depending on which pass it was obtained from
def getTractList(col):
    coordinates = []
    tract_list = [] 
    
    # Get the coordinates from the most specific pass
    if (col['Explicit_Pass_Coordinates'] != None): # From title
        coordinates = col['Explicit_Pass_Coordinates'][1]
    elif(col['NER_Pass_Coordinates'] != None): # From NER Pass
        coordinates = col['NER_Pass_Coordinates']
    elif (col['NER_Pred_Coordinates'] != None): # From LLM Pass
        coordinates = col['NER_Pred_Coordinates']
    else: # Must be a very hard/bad article :-(
        return None 
        
    longitude, latitude = coordinates
    TRACT, COUNTY = query_census_api(longitude,latitude)
    tract_list.append(TRACT)
    
    return tract_list

In [412]:
df['Tracts'] = df.progress_apply(getTractList, axis=1)

100%|██████████| 10/10 [00:03<00:00,  2.62it/s]


In [403]:
df

,_id,hl1,body,llama_prediction,Explicit_Pass_1,NER_Pass_1,NER_Pass_1_Sorted,NER_Pass_1_Coordinates,NER_prediction,NER_Sorted,NER_Pred_Sorted,NER_Pred_Coordinates,Tracts
8342,0000017f-fa76-d9c5-a37f-fff6a0c40000,LIVESTREAM Boston Public Radio live from the G...,,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...",None,None,None,None,None,None,None,[No TRACT found]
11325,00000185-0634-d270-adb5-57b7d9440001,Karen Bass is sworn in as Los Angeles mayor as...,Karen Bass was sworn in as the mayor of Los An...,Based on the information provided in the art...,None,"[((Karen, Bass), PERSON), ((Los, Angeles), GPE...","[((Los, Angeles, City, Council), ORG), ((NPR),...",None,"[((1), CARDINAL), ((Los, Angeles), GPE), ((Los...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[((City, Hall), FAC), ((City, Hall), FAC), ((L...","[-70.9078346, 42.3020647]",[500101]
9947,00000182-a72b-d8f2-a1fa-efaf3de50001,More than million baby swings and rockers are ...,4moms is recalling more than million of its Ma...,Response:\n\n1. Y - The article is talking a...,None,"[((4moms), ORG), ((more, than, million), CARDI...","[((4moms), ORG), ((the, Consumer, Product, Saf...",None,"[((1), CARDINAL), ((Boston), GPE), ((Massachus...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[((BuyBuy, Baby), ORG), ((Target), ORG), ((Ama...","[-71.3824374, 42.4072107]",[365100]
4320,0000017a-0ca7-d770-a57f-edbf9b070001,We re Just Asking For Some Compassion Tenants ...,When the pandemic first hit Dezyre Lewis found...,None,None,"[((first), ORDINAL), ((Dezyre, Lewis), PERSON)...","[((Residential, Assistance, for, Families, in,...",None,None,None,None,None,None
1376,00000176-d8be-de44-a77e-febe4a4b0001,Pressley Moulton Call For Trump Removal After ...,As pro Trump extremists began to recede from t...,None,None,"[((Trump), PERSON), ((U.S.), GPE), ((Capitol),...","[((Capitol), FAC), ((Capitol), FAC), ((the, Ho...","[-71.0638101, 42.3587772]",None,None,None,None,[020302]
8076,0000017f-9d68-d191-a1ff-fd6869480001,We can always strive to be more inclusive Miss...,The Miss America contest has been around for m...,"Based on the article, I would guess that the...",None,"[((Miss, America), WORK_OF_ART), ((more, than,...","[((Miss, America), ORG), ((Miss, America), ORG...",None,"[((Cambridge), GPE), ((Massachusetts), GPE), (...","[((Miss, America), ORG), ((Cambridge), GPE), (...","[((Miss, America), ORG), ((Cambridge), GPE), (...","[-71.3824374, 42.4072107]",[365100]
5228,0000017b-410a-d5be-a57b-fd8a3b1d0001,Friday August 20,Discover America creative spirit through the l...,Based on the information provided in the art...,None,"[((America), GPE), ((American), NORP), ((Rhian...","[((GBH, Create), ORG), ((Backcredit, Craft, in...",None,"[((Boston), GPE), ((Massachusetts), GPE), ((1)...","[((GBH, Create), ORG), ((Craft, in, America, I...","[((GBH, Create), ORG), ((Craft, in, America, I...","[-71.3824374, 42.4072107]",[365100]
12382,00000186-9287-d649-ab9f-bf87b3830001,Scientists are flying into snowstorms to explo...,High up in some ice filled clouds sitting insi...,"Based on the article, I would guess that the...",None,"[((Christian, Nairy), PERSON), ((Nairy), PERSO...","[((the, University, of, North, Dakota), ORG), ...",None,"[((the, United, States), GPE), ((Virginia), GP...","[((Wallops, Flight, Facility), FAC), ((NASA, W...","[((Wallops, Flight, Facility), FAC), ((NASA, W...","[-71.3824374, 42.4072107]",[365100]
4666,0000017a-8140-d2d2-a1fe-d3e93c780001,Addressing Mental Health Concerns In The Resta...,While the pandemic has impacted the mental hea...,"1. Y - Yes, the article is talking about a r...",None,"[((McLean, Hospital), ORG), ((Belmont), GPE), ...","[((McLean, Hospital), ORG), ((RISE, Hospitalit...",None,"[((1), CARDINAL), ((Boston), GPE), ((2), CARDI...","[((McLean, Hospital), ORG), ((RISE, Hospitalit...","[((McLean, Hospital), ORG), ((RISE, Hospitalit...","[-71.1907002, 42.3943215]",[357700]
7916,0000017f-6ae7-d19f-a3ff-eff72efd0001,Disney employees furious the co

In [ ]:
df = df.dropna(subset=["Tracts"]) # Clean those that don't have a Tract

In [ ]:
df

## Topic Modeling

In [13]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [14]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [ ]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [ ]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns